# Data Preparation with xaytune

xaytune includes a comprehensive data preparation toolkit for fine-tuning datasets. It supports:

- **Format conversion**: Convert between Alpaca, ShareGPT, CSV, JSONL, and other formats
- **Quality filtering**: Remove low-quality samples using length, language, regex, and custom filters
- **Deduplication**: Exact and fuzzy (MinHash) deduplication to remove redundant samples
- **Synthetic generation**: Generate training data via augmentation, distillation, or evolution
- **Pipelines**: Chain multiple operations together via Python API or YAML config

This notebook demonstrates all these features with working examples.

In [ ]:
# Install xaytune with data prep extras
# pip install xaytune[data-prep]  # for dedup and language filtering
# pip install xaytune[synth]      # for synthetic data generation
# pip install xaytune[data-all]   # everything

## Sample Data Setup

Let's create some sample data to work with. We'll use the Alpaca format (instruction, input, output).

In [ ]:
samples = [
    {"instruction": "What is machine learning?", "input": "", "output": "Machine learning is a subset of AI that enables systems to learn from data."},
    {"instruction": "What is machine learning?", "input": "", "output": "Machine learning is a subset of AI that enables systems to learn from data."},  # duplicate
    {"instruction": "Hi", "input": "", "output": "Hey"},  # too short
    {"instruction": "Explain neural networks", "input": "", "output": "Neural networks are computing systems inspired by biological neural networks that consist of interconnected nodes."},
    {"instruction": "Visit my site", "input": "", "output": "Check out https://spam.com for deals"},  # spam URL
    {"instruction": "What is deep learning?", "input": "", "output": "Deep learning uses multi-layer neural networks to learn hierarchical representations of data."},
]

print(f"Created {len(samples)} sample records")

## Format Conversion

Convert between different dataset formats. xaytune supports Alpaca, ShareGPT, CSV, JSONL, and custom formats with field mapping.

In [ ]:
from xaytune.data.prep import convert

result = convert(samples, source_format="alpaca", target_format="sharegpt")
print(f"Converted {len(result.dataset)} samples to ShareGPT format")
print("\nFirst sample in ShareGPT format:")
print(result.dataset[0])

## Quality Filtering

Filter out low-quality samples using built-in filters:

- **length**: Minimum/maximum character or word count
- **regex**: Drop samples matching a pattern (e.g., spam URLs)
- **language**: Keep only specific languages (requires `langdetect`)
- **custom**: Register your own filter functions

In [ ]:
from xaytune.data.prep import filter_dataset

result = filter_dataset(
    samples,
    filters=[
        {"type": "length", "min_chars": 20},
        {"type": "regex", "drop_pattern": r"https?://\S+"},
    ],
    field="output",
)

print(result.report.summary())
print(f"\nKept {len(result.dataset)} of {len(samples)} samples")

## Custom Filters

Register custom filter functions for domain-specific quality checks.

In [ ]:
from xaytune.data.prep import register_filter

@register_filter("no-greeting")
def drop_greetings(sample, field):
    """Drop samples that are just greetings."""
    greetings = {"hi", "hey", "hello", "bye", "goodbye"}
    return sample[field].strip().lower() not in greetings

result = filter_dataset(
    samples,
    filters=[{"type": "no-greeting"}],
    field="output",
)

print(f"After greeting filter: {len(result.dataset)} samples")
print(result.report.summary())

## Deduplication

Remove duplicate samples using exact matching or fuzzy MinHash-based similarity detection.

- **exact**: Remove exact duplicates (fast)
- **minhash**: Remove similar samples using locality-sensitive hashing
- **both**: Run both methods sequentially

In [ ]:
from xaytune.data.prep import deduplicate

result = deduplicate(samples, method="exact", field="output")
print(result.report.summary())
print(f"\nRemoved {len(samples) - len(result.dataset)} duplicate(s)")
print(f"Final dataset: {len(result.dataset)} samples")

## Pipeline

Chain multiple operations together in a single pipeline. Operations are applied in sequence, with stats tracked at each step.

In [ ]:
from xaytune.data.prep import pipeline

result = pipeline(
    input=samples,
    steps=[
        {"filter": {"min_chars": 20}},
        {"deduplicate": {"method": "exact"}},
    ],
)

print(result.report.summary())
print(f"\nFinal dataset: {len(result.dataset)} samples")
print("\nSamples:")
for s in result.dataset:
    print(f"  - {s['instruction'][:50]}...")

## YAML Pipeline Configuration

For complex pipelines, you can use YAML configuration files. This is especially useful for reproducible data prep workflows.

In [ ]:
yaml_config = """input: data/train.jsonl
output: data/train_clean.jsonl
steps:
  - convert:
      source_format: csv
      target_format: alpaca
      field_map:
        question: instruction
        answer: output
  - filter:
      min_chars: 50
      language: en
  - deduplicate:
      method: both
      threshold: 0.85
"""

print("Example pipeline YAML config:")
print(yaml_config)
print("\nRun with: xaytune data pipeline prep_config.yaml")

## Synthetic Data Generation

Generate synthetic training data using LLMs. Supports three modes:

- **augment**: Generate variations of seed examples
- **distill**: Generate examples from a topic description
- **evolve**: Make examples progressively more complex/difficult

Works with any OpenAI-compatible API (OpenAI, Anthropic, vLLM, Ollama, etc.).

In [ ]:
# Synthetic data generation requires an API key
# Works with any OpenAI-compatible endpoint (OpenAI, Anthropic, vLLM, Ollama)

# from xaytune.data.prep import generate
#
# # Augment: generate variations of seed examples
# result = generate(
#     mode="augment",
#     seed=samples[:2],
#     n=100,
#     format="alpaca",
#     model="gpt-4o-mini",
#     api_key="sk-...",  # or set OPENAI_API_KEY env var
# )
#
# # Distill: generate from a topic description
# result = generate(
#     mode="distill",
#     topic="Python debugging techniques",
#     n=50,
#     format="alpaca",
#     model="gpt-4o-mini",
# )
#
# # Evolve: make examples progressively harder
# result = generate(
#     mode="evolve",
#     seed=samples[:2],
#     rounds=3,
#     format="alpaca",
#     model="gpt-4o-mini",
# )
#
# print(f"Generated {len(result.dataset)} synthetic samples")

## Recipe Integration

Data preparation pipelines can be integrated directly into training configs. The `data_prep` section will run automatically before training.

In [ ]:
train_config = """recipe: finetune
model:
  name: meta-llama/Llama-3.1-8B
dataset: data/raw.csv
data_prep:
  - convert:
      source_format: csv
      target_format: alpaca
      field_map:
        question: instruction
        answer: output
  - filter:
      min_chars: 50
  - deduplicate:
      method: both
training:
  num_epochs: 3
  batch_size: 4
"""

print("Training config with data_prep:")
print(train_config)
print("\nData prep runs automatically before training starts.")

## CLI Reference

The data prep toolkit is also available via the command line:

```bash
# Deduplicate
xaytune data deduplicate data.jsonl -o clean.jsonl --method both --threshold 0.85

# Filter
xaytune data filter data.jsonl -o filtered.jsonl --min-chars 50 --drop-regex 'https?://\S+'

# Convert
xaytune data convert data.jsonl -o converted.jsonl --from alpaca --to sharegpt

# Generate synthetic data
xaytune data generate --mode distill --topic "Python basics" -n 100 --model gpt-4o-mini -o synthetic.jsonl

# Run a pipeline from YAML
xaytune data pipeline prep_config.yaml
```

All commands support `--help` for detailed options.

## Next Steps

Now that you know how to prepare data, check out these notebooks:

- **01_quickstart.ipynb**: Get started with xaytune basics
- **02_finetuning.ipynb**: Train models with your prepared data
- **03_gpu_training.ipynb**: Scale up training with GPUs

For more details, see the [data preparation documentation](https://github.com/szaher/xaytune#data-preparation).